## Report Prompts for Media Diversity Watch Agent

These prompts live in the **Report node** of your LangGraph workflow.

The report is built in 4 steps. Each step calls the LLM once.
Each step feeds its output into the next step.
This way the LLM is never asked to do too much at once --
it focuses on one job at a time, which produces better output.

```
Step 1: ANGLE_PROMPT       -- writes the 4 evaluation angle sections
Step 2: COMMUNITY_PROMPT   -- writes one summary per community (4 paragraphs)
Step 3: SYNTHESIS_PROMPT   -- generates scores, strengths, harm flags
Step 4: REPORT_PROMPT      -- assembles the complete 11-section report
```

**Why 4 steps instead of 1?**

If you ask the LLM to write the entire report in one call,
it gets overwhelmed and starts hallucinating or writing generic filler.
Breaking it into 4 steps means each call is focused and short.
The quality is dramatically better.

**Key rule in every prompt:**
Every claim must cite its source in [square brackets].
This is how we prevent hallucination -- the LLM can only use
what was actually returned by the tools.

In [ ]:
# ── CELL: IMPORT AND DATE SETUP ─────────────────────────────────
# Run this first. It sets up the date variables used in the prompts.

from datetime import datetime

today = datetime.now()

print(f"Today: {today.strftime('%A %d %B %Y')}")
print(f"Week number: {today.strftime('%W')}")
print(f"Year: {today.year}")

## Step 1: Angle Prompt

This writes the four evaluation angle sections.

What it produces:
- Angle 1: Bylines and Story Selection
- Angle 2: Portrayal Within Content
- Angle 3: Sourcing Diversity
- Angle 4: Language and Framing

Each section is 2 to 4 sentences of analytical prose with source citations.
No bullet points. No raw numbers without context.

In [ ]:
# ── CELL: ANGLE PROMPT ──────────────────────────────────────────
# Writes the four evaluation angle sections.
# Input:  company name, date, all tool results
# Output: 4 prose sections, each 2-4 sentences, all claims cited

ANGLE_PROMPT = """
You are Nova_Tracks, lead researcher at Media Diversity Watch.
You are writing the weekly monitoring report for {company}.

Your job right now: write FOUR EVALUATION ANGLE sections.
Each section must be 2 to 4 sentences of clear, analytical prose.

TONE: Neutral. Evidence-based. Direct.
Write like a researcher briefing colleagues -- not a newspaper headline.
No dramatic language. No unsupported generalisations.

RULES:
- Write in full sentences. No bullet points inside sections.
- Every statistic or finding must be followed by its source in [square brackets].
  Example: "Women appeared in 4 of 28 articles (14%) [RSS Feed: {company}, {date}]."
- If no evidence was found for a community or angle, write exactly this:
  No evidence found in this monitoring period.
- Do not invent numbers. Do not round up findings to sound more significant.
- Do not use your training data. Only use the evidence provided below.

GOOD EXAMPLE (write like this):
BBC published 28 articles in the monitored period. Women were mentioned as
subjects or authors in 4 of those articles (14%), which falls below the 37%
female byline benchmark in industry research [Pinecone RAG: Nieman Lab 2023].
Coverage of racial minority communities appeared in 3 articles, all in crime
or conflict contexts rather than leadership or expertise [RSS Feed: BBC, {date}].

BAD EXAMPLE (do not write like this):
- Women: 4 articles, 14%
- Race: 3 articles
- Score: low

---
EVIDENCE FROM RESEARCH TOOLS:
{tool_results}
---

Write these four sections now:

## Angle 1: Bylines and Story Selection
Write 2 to 4 sentences about who is writing stories and what stories are being
chosen. Cite specific counts from RSS and NewsAPI results.

## Angle 2: Portrayal Within Content
Write 2 to 4 sentences about how communities are framed in the content.
Are they shown as experts and leaders, or as victims and problems?
Cite specific article titles or language examples where found.

## Angle 3: Sourcing Diversity
Write 2 to 4 sentences about who gets quoted as an authority in stories.
Are community members speaking for themselves or being spoken about?
Cite any evidence from tool results.

## Angle 4: Language and Framing
Write 2 to 4 sentences about the language used in headlines and descriptions.
Flag any specific terms that are non-inclusive, with the exact headline as evidence.
If the language audit found no flags, state that clearly.
"""

print("ANGLE_PROMPT loaded.")
print(f"Variables to fill: company, date, tool_results")
print(f"Output: 4 prose sections")

## Step 2: Community Prompt

This writes one summary paragraph for each of the four communities.
It draws on the angle analysis from Step 1 and synthesises across all angles.

This is the section that reads most like a real research briefing --
it answers the question: "So what does this mean for women specifically?
For LGBTQ+ communities specifically?"

In [ ]:
# ── CELL: COMMUNITY PROMPT ───────────────────────────────────────
# Writes one paragraph per community synthesising across all 4 angles.
# Input:  company name, date, angle_analysis from Step 1, tool results
# Output: 4 paragraphs (3-5 sentences each), one per community

COMMUNITY_PROMPT = """
You are Nova_Tracks, lead researcher at Media Diversity Watch.
You are writing the per-community section of the weekly monitoring report for {company}.

Your job: write one paragraph of 3 to 5 sentences for each of the four communities.
Each paragraph summarises how {company} covered that community this week,
drawing on findings from all four evaluation angles.

TONE: Neutral. Evidence-based. Direct.

RULES:
- Write in full sentences. No bullet points.
- Each paragraph must reference at least one specific finding from the evidence.
- Cite sources in [square brackets] for every statistic.
- If a community was barely or not represented, say so plainly and concisely.
- Do not copy the angle sections word for word. This is a synthesis, not a repeat.
- Do not use your training data. Only use the evidence provided below.

GOOD EXAMPLE:
Coverage of LGBTQ+ communities was minimal this week. One article mentioned
a same-sex couple in a brief human interest piece, but no LGBTQ+ voices appeared
as expert sources or authors in any monitored content [RSS Feed: {company}, {date}].
No language flags were raised for this community, but the near-absence of coverage
itself reflects a pattern documented in our research base [Pinecone RAG: GLAAD 2023].

---
ANGLE ANALYSIS (from Step 1):
{angle_analysis}

ADDITIONAL EVIDENCE FROM TOOLS:
{tool_results}
---

## Women and Gender Equality
Write 3 to 5 sentences summarising this week's findings for this community
across all 4 evaluation angles.

## LGBTQ+ Communities
Write 3 to 5 sentences summarising this week's findings for this community
across all 4 evaluation angles.

## Racial and Ethnic Minorities
Write 3 to 5 sentences summarising this week's findings for this community
across all 4 evaluation angles.

## People with Disabilities
Write 3 to 5 sentences summarising this week's findings for this community
across all 4 evaluation angles.
"""

print("COMMUNITY_PROMPT loaded.")
print(f"Variables to fill: company, date, angle_analysis, tool_results")
print(f"Output: 4 community paragraphs")

## Step 3: Synthesis Prompt

This generates the scores, strengths, and harm flags.

The scores are not just numbers -- each score comes with one sentence of reasoning
so the number has meaning. A 4/10 with no explanation is useless.
A 4/10 with 'representation appeared in 14% of articles against a 37% benchmark'
is evidence.

In [ ]:
# ── CELL: SYNTHESIS PROMPT ───────────────────────────────────────
# Generates scores per community, strengths section, and harm flags.
# Input:  company, angle_analysis, community_summaries, all tool results
# Output: structured scores with reasoning, prose strengths, prose/list harm flags

SYNTHESIS_PROMPT = """
You are Nova_Tracks, lead researcher at Media Diversity Watch.
You are writing the synthesis section of the weekly monitoring report for {company}.

Your job: produce three things.

1. SCORES
Assign a score from 1 to 10 for each of the four communities.
Base scores only on the evidence provided. No impressions.

Scoring guide:
1 to 3 = significant concerns or near absence of representation
4 to 6 = mixed, some representation with notable gaps
7 to 9 = generally positive with minor issues
10 = exemplary (rare, requires strong evidence across all angles)

After each score, write ONE sentence explaining the reasoning.
Example: Women: 5/10 -- Representation appeared in 4 of 28 articles (14%),
below the 37% industry benchmark, though two articles featured women as expert sources.

2. STRENGTHS
Write 2 to 4 sentences identifying what {company} did well this week.
Be specific. Cite the evidence.
If nothing stands out, write exactly:
No notable strengths identified in this monitoring period.

3. HARM FLAGS
List any specific language or framing that caused active harm.
For each flag, quote the exact headline or phrase as evidence.
If none were found, write exactly:
No harm flags identified in this monitoring period.

RULES:
- Do not invent scores. Base them on evidence only.
- Do not use your training data. Only use what is provided below.
- Strengths section must be written in prose sentences, not a list.

---
ANGLE ANALYSIS:
{angle_analysis}

COMMUNITY SUMMARIES:
{community_summaries}

ALL TOOL RESULTS:
{tool_results}
---

Return your response in this exact format:

SCORES:
Women and Gender Equality: [score]/10 -- [one sentence reasoning with source]
LGBTQ+ Communities: [score]/10 -- [one sentence reasoning with source]
Racial and Ethnic Minorities: [score]/10 -- [one sentence reasoning with source]
People with Disabilities: [score]/10 -- [one sentence reasoning with source]
Overall: [average of the four scores, rounded to 1 decimal]/10

STRENGTHS:
[2 to 4 sentences of prose]

HARM FLAGS:
[One line per flag: Flag: "[exact quote]" -- [brief explanation of why this is flagged]]
[Or: No harm flags identified in this monitoring period.]
"""

print("SYNTHESIS_PROMPT loaded.")
print(f"Variables to fill: company, angle_analysis, community_summaries, tool_results")
print(f"Output: scores with reasoning, strengths prose, harm flags")

## Step 4: Final Report Prompt

This assembles everything into the complete 11-section report.

It also writes two new sections that need the full picture to write well:
- Company Overview (from Wikipedia)
- Recommendations (based on the worst-scoring findings)

The output of this prompt is the final markdown file that gets
saved to /reports/ and sent to Notion.

In [ ]:
# ── CELL: FINAL REPORT PROMPT ────────────────────────────────────
# Assembles the complete 11-section report.
# Input:  all previous step outputs + wikipedia result + sources list
# Output: full markdown report, ready to save and send to Notion

REPORT_PROMPT = """
You are Nova_Tracks, lead researcher at Media Diversity Watch.
Assemble the full weekly monitoring report for {company}.
Week {week_number}, {year}.

You have all the component parts below.
Your job:
1. Write a short Company Overview section from the Wikipedia background.
2. Insert the pre-written angle and community sections exactly as provided.
3. Insert the scores, strengths, and harm flags exactly as provided.
4. Write 3 to 5 specific, actionable recommendations based on the evidence.
5. Compile the full source list from all tool calls.

TONE: Neutral. Evidence-based. Written for a researcher audience.
Concise but complete. Each section should feel like a paragraph in a
research briefing, not a spreadsheet printout.

FORMAT:
Use markdown headers. Write in prose. No nested bullet points inside sections.
Source list at the end may use a numbered list.

RECOMMENDATIONS RULES:
Be specific. Not just 'improve diversity' but 'increase female bylines in
political coverage, where the current ratio is X% against an industry average of Y%.'
Ground every recommendation in a specific finding from this report.
Write 3 to 5 recommendations, each 1 to 2 sentences.

---
COMPANY BACKGROUND (Wikipedia):
{wikipedia_result}

ANGLE ANALYSIS (Step 1):
{angle_analysis}

COMMUNITY SUMMARIES (Step 2):
{community_summaries}

SCORES, STRENGTHS, FLAGS (Step 3):
{synthesis}

ALL SOURCES USED:
{sources}
---

Write the full report now:

# {company} Inclusivity Report
**Media Diversity Watch | Week {week_number}, {year}**

---

## 1. Company Overview
Write 2 to 3 sentences from the Wikipedia background.
Include ownership, size, and editorial stance.
Add one sentence noting any publicly stated diversity commitments if found.

---

## 2. Bylines and Story Selection
[Insert Angle 1 section from angle_analysis exactly as written]

## 3. Portrayal Within Content
[Insert Angle 2 section from angle_analysis exactly as written]

## 4. Sourcing Diversity
[Insert Angle 3 section from angle_analysis exactly as written]

## 5. Language and Framing
[Insert Angle 4 section from angle_analysis exactly as written]

---

## 6. Community-by-Community Findings

### Women and Gender Equality
[Insert from community_summaries]

### LGBTQ+ Communities
[Insert from community_summaries]

### Racial and Ethnic Minorities
[Insert from community_summaries]

### People with Disabilities
[Insert from community_summaries]

---

## 7. Strengths
[Insert from synthesis]

## 8. Harm Flags
[Insert from synthesis]

## 9. Inclusivity Scores

[Insert scores from synthesis]

Write 2 to 3 sentences of context around the overall score.
What does it mean? Is this above or below what we typically see?
What is the most urgent gap?

---

## 10. Recommendations
Write 3 to 5 specific, evidence-based recommendations.
Each recommendation should be 1 to 2 sentences.
Root every recommendation in a finding from this report.

---

## 11. Sources
Numbered list of all sources cited in this report.
Format each as: [Source type]: [description] -- [tool that provided it]
Example:
1. RSS Feed: BBC World News homepage feed, 28 articles analysed -- analyse_rss_feed
2. NewsAPI: 7 articles about BBC diversity coverage -- search_news
3. Pinecone RAG: Nieman Lab 2023 byline diversity study -- rag_query
"""

print("REPORT_PROMPT loaded.")
print(f"Variables to fill: company, week_number, year,")
print(f"  wikipedia_result, angle_analysis, community_summaries, synthesis, sources")
print(f"Output: full 11-section markdown report")

## How to Use These Prompts in the Report Node

When you build the Report node in `day3_agent.ipynb`,
call the prompts in sequence like this:

In [ ]:
# ── CELL: REPORT NODE TEMPLATE ───────────────────────────────────
# This is the Report node function for your LangGraph workflow.
# Copy this into day3_agent.ipynb when you build the agent.
#
# It calls all 4 prompts in sequence and stores the final report
# in state["report_text"] ready for N8N to pick up and send to Notion.

def report_node(state: dict) -> dict:
    """
    Report node: assembles the full 11-section inclusivity report.
    Calls 4 LLM prompts in sequence.
    Each step feeds its output into the next.
    Returns state with report_text filled in.
    """
    from datetime import datetime

    company    = state["company"]
    today      = datetime.now()
    date_str   = today.strftime("%d %B %Y")
    week_num   = today.strftime("%W")
    year       = today.year

    # Collect all tool results into one string for the prompts
    tool_results = "\n\n".join([
        f"--- {result['tool']} ---\n{result['output']}"
        for result in state.get("tool_results", [])
    ])

    # Extract Wikipedia result separately for the overview section
    wikipedia_result = next(
        (r["output"] for r in state.get("tool_results", [])
         if r["tool"] == "get_wikipedia"),
        "No Wikipedia data retrieved."
    )

    # Build source list from all tools called
    sources = "\n".join([
        f"{i+1}. {r['tool']}: {r.get('query', 'See output')}"
        for i, r in enumerate(state.get("tool_results", []))
    ])

    print(f"[REPORT] Step 1: Writing angle sections for {company}...")

    # Step 1: Write the 4 angle sections
    angle_result = llm.invoke(
        ANGLE_PROMPT.format(
            company=company,
            date=date_str,
            tool_results=tool_results
        )
    )
    angle_analysis = angle_result.content

    print(f"[REPORT] Step 2: Writing community summaries...")

    # Step 2: Write the 4 community summaries
    community_result = llm.invoke(
        COMMUNITY_PROMPT.format(
            company=company,
            date=date_str,
            angle_analysis=angle_analysis,
            tool_results=tool_results
        )
    )
    community_summaries = community_result.content

    print(f"[REPORT] Step 3: Generating scores and synthesis...")

    # Step 3: Generate scores, strengths, harm flags
    synthesis_result = llm.invoke(
        SYNTHESIS_PROMPT.format(
            company=company,
            angle_analysis=angle_analysis,
            community_summaries=community_summaries,
            tool_results=tool_results
        )
    )
    synthesis = synthesis_result.content

    # Parse overall score from synthesis output
    overall_score = 0.0
    for line in synthesis.split("\n"):
        if line.startswith("Overall:"):
            try:
                # Extract the number before /10
                score_str = line.split(":")[1].strip().split("/")[0].strip()
                overall_score = float(score_str)
            except Exception:
                overall_score = 0.0
            break

    print(f"[REPORT] Step 4: Assembling full report...")

    # Step 4: Assemble the complete report
    final_report = llm.invoke(
        REPORT_PROMPT.format(
            company=company,
            week_number=week_num,
            year=year,
            wikipedia_result=wikipedia_result,
            angle_analysis=angle_analysis,
            community_summaries=community_summaries,
            synthesis=synthesis,
            sources=sources
        )
    )
    report_text = final_report.content

    # Save report to file
    import os
    os.makedirs("../reports", exist_ok=True)
    report_filename = f"../reports/{company.lower().replace(' ', '_')}_week{week_num}_{year}.md"

    with open(report_filename, "w") as f:
        f.write(report_text)

    print(f"[REPORT] Complete. Saved to {report_filename}")
    print(f"[REPORT] Overall score: {overall_score}/10")

    # Update state with everything N8N needs
    state["report_text"]   = report_text
    state["report_path"]   = report_filename
    state["overall_score"] = overall_score

    return state


print("report_node() function defined.")
print("Ready to add to LangGraph workflow in day3_agent.ipynb")

## Quick Test: Check Prompts Load and Format Correctly

Run this cell to verify the prompts fill in correctly with sample data.
This does NOT call the LLM -- it just checks the variables slot in.

In [ ]:
# ── CELL: PROMPT FORMAT TEST ─────────────────────────────────────
# Verifies all prompts format correctly with dummy data.
# Does NOT call the LLM. Safe to run any time.

test_company     = "BBC"
test_date        = "27 February 2026"
test_tool_result = "[DUMMY] 28 articles analysed. Women: 4. LGBTQ: 1. Race: 3. Disability: 0."
test_angle       = "[DUMMY ANGLE ANALYSIS]"
test_community   = "[DUMMY COMMUNITY SUMMARIES]"
test_synthesis   = "Overall: 4.5/10"
test_sources     = "1. RSS Feed: BBC\n2. NewsAPI: BBC diversity"
test_wiki        = "BBC is a British public broadcaster founded in 1927."

# Test Step 1
formatted_angle = ANGLE_PROMPT.format(
    company=test_company,
    date=test_date,
    tool_results=test_tool_result
)
print("ANGLE_PROMPT: OK" if test_company in formatted_angle else "ANGLE_PROMPT: ERROR")

# Test Step 2
formatted_community = COMMUNITY_PROMPT.format(
    company=test_company,
    date=test_date,
    angle_analysis=test_angle,
    tool_results=test_tool_result
)
print("COMMUNITY_PROMPT: OK" if test_company in formatted_community else "COMMUNITY_PROMPT: ERROR")

# Test Step 3
formatted_synthesis = SYNTHESIS_PROMPT.format(
    company=test_company,
    angle_analysis=test_angle,
    community_summaries=test_community,
    tool_results=test_tool_result
)
print("SYNTHESIS_PROMPT: OK" if test_company in formatted_synthesis else "SYNTHESIS_PROMPT: ERROR")

# Test Step 4
formatted_report = REPORT_PROMPT.format(
    company=test_company,
    week_number="09",
    year=2026,
    wikipedia_result=test_wiki,
    angle_analysis=test_angle,
    community_summaries=test_community,
    synthesis=test_synthesis,
    sources=test_sources
)
print("REPORT_PROMPT: OK" if test_company in formatted_report else "REPORT_PROMPT: ERROR")

print("")
print("All 4 prompts ready for Day 3 agent notebook.")